# Presentation figures — sweep results, one claim per chart

Rebuilds the four deck figures (plus one backup) from the campaign's
metric CSVs only (`results/*_results_v2.csv` + `results/*_horizon.csv`;
the large per-cycle prediction files are not needed). Figures are saved
to `results/figures/deck/` as PNG + PDF.

All annotations (equivalence arrows, ratios, failure counts) are
**computed from the loaded data at run time** — nothing is hardcoded.

Selection rules, for provenance:
- Headline dataset is **FD002** — the *hardest* C-MAPSS variant
  (6 operating conditions, 260 units), not the most flattering one;
  the backup cell shows the same pattern on FD001/FD003/FD004.
- The comparison line is the **per-point best of all five from-scratch
  arms** (GBM, GBM+age, LSTM, CNN, MiniRocket) — the hardest baseline.
- The TSFM arm is **CORN at every point** (the a-priori deployment
  recommendation), even where MSE happens to be lower.
- N-CMAPSS and XJTU-SY figures use the **all-cycles horizon eval**;
  the last-cycle protocol is degenerate there (fixed 0.6 truncation
  makes the mean predictor near-optimal on 3–6 test predictions).

## Setup — mount Drive and point at `results/`

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:  # Colab: mount Drive; anywhere else: skip silently
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    RESULTS_DIR = Path("/content/drive/MyDrive/Predictive Maintenance LSTM/"
                       "Predictive-Maintenance-LSTM/results")
except ImportError:
    RESULTS_DIR = Path(os.environ.get("RESULTS_DIR", "../results"))

OUT_DIR = RESULTS_DIR / "figures" / "deck"
OUT_DIR.mkdir(parents=True, exist_ok=True)
HEADLINE_DATASET = "FD002"   # hardest C-MAPSS variant: 6 operating conditions
print("results:", RESULTS_DIR, "| figures ->", OUT_DIR)

## Chart style (validated palette, recessive grid)

In [ ]:
BLUE, YELLOW, GREEN, ORANGE = "#2a78d6", "#eda100", "#008300", "#eb6834"
INK, INK2, MUTED, GRID = "#0b0b0b", "#52514e", "#9a978f", "#e6e4df"
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": GRID, "axes.linewidth": 0.8,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.7,
    "axes.axisbelow": True, "font.size": 11, "axes.titlesize": 14,
    "axes.labelsize": 11, "text.color": INK, "axes.labelcolor": INK2,
    "xtick.color": INK2, "ytick.color": INK2,
    "xtick.labelsize": 10.5, "ytick.labelsize": 10.5,
})

def style(ax, ygrid_only=True):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    if ygrid_only:
        ax.grid(axis="y"); ax.grid(axis="x", visible=False)

def save(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(OUT_DIR / f"{name}.{ext}", bbox_inches="tight", dpi=200)
    print("saved", OUT_DIR / f"{name}.png")

## Load the metric CSVs

In [ ]:
res = pd.concat([pd.read_csv(f) for f in sorted(RESULTS_DIR.glob("*_results_v2.csv"))],
                ignore_index=True)
res["arm"] = res["model"] + np.where(res["loss"].isin(["native"]) | res["loss"].isna(),
                                     "", "/" + res["loss"].astype(str))
hz = pd.concat([pd.read_csv(f) for f in sorted(RESULTS_DIR.glob("*_horizon.csv"))],
               ignore_index=True)
print(f"results_v2 rows: {len(res)} | horizon rows: {len(hz)} | "
      f"datasets: {sorted(res.dataset.unique())}")

## Fig 1 — Data efficiency (FD002 scaling curve)

In [ ]:
# Fig 1 -- data efficiency. Chronos+CORN (deployment recommendation, fixed a
# priori) vs the STRONGEST from-scratch arm at each fleet size (per-point best
# of all five: the hardest comparison), plus the predict-mean floor.
BASELINE_ARMS = ["gbm", "gbm_age", "lstm", "cnn", "minirocket"]
d = res[res.dataset == HEADLINE_DATASET]
counts = sorted(d.n_units.unique())
g = d.groupby(["arm", "n_units"]).rmse_clipped.agg(["mean", "std"])
ch_m = g.loc["chronos-2_mlp/corn"]["mean"].reindex(counts)
ch_s = g.loc["chronos-2_mlp/corn"]["std"].reindex(counts)
bl = (d[d.arm.isin(BASELINE_ARMS)].groupby(["arm", "n_units"]).rmse_clipped.mean()
      .groupby("n_units").min().reindex(counts))
floor = g.loc["predict_mean"]["mean"].reindex(counts)

# data-driven equivalence annotation: how many units do from-scratch models
# need to match the TSFM trained on `anchor` units?
anchor = counts[1]
below = [n for n in counts if bl[n] <= ch_m[anchor]]
n_hi = below[0] if below else counts[-1]
n_lo = counts[max(0, counts.index(n_hi) - 1)]

fig, ax = plt.subplots(figsize=(9.2, 5.6), dpi=110)
x = np.arange(len(counts))
ax.fill_between(x, ch_m - ch_s, ch_m + ch_s, color=BLUE, alpha=0.13, lw=0)
ax.plot(x, ch_m, color=BLUE, lw=2.4, marker="o", ms=6, zorder=5,
        label="Foundation model (frozen Chronos-2 + small head)")
ax.plot(x, bl.values, color=YELLOW, lw=2.0, ls="--", marker="s", ms=5.5, zorder=4,
        label="Strongest from-scratch model at each fleet size")
ax.plot(x, floor.values, color=MUTED, lw=1.3, ls=":", zorder=3)
ax.annotate("naive floor: always predict the fleet average",
            xy=(0.0, floor.iloc[0] - 1.0), color=MUTED, fontsize=10, va="top")
y5 = float(ch_m[anchor])
ax.annotate("", xy=(x[counts.index(n_hi)] + 0.12, y5), xytext=(x[1] + 0.12, y5),
            arrowprops=dict(arrowstyle="->", color=INK2, lw=1.2))
ax.annotate(f"same accuracy with {anchor} failure histories\n"
            f"as from-scratch models need {n_lo}–{n_hi}",
            xy=(x[2] + 0.55, y5), xytext=(x[2] + 0.35, y5 + 6.5), fontsize=10.5,
            color=INK, ha="center",
            arrowprops=dict(arrowstyle="-", color=INK2, lw=0.8))
ax.annotate("foundation\nmodel", xy=(x[-1], ch_m.iloc[-1]),
            xytext=(x[-1] + 0.12, ch_m.iloc[-1] - 1.2), color=BLUE,
            fontsize=10.5, fontweight="bold", va="top")
ax.annotate("from-scratch\nbest", xy=(x[-1], bl.iloc[-1]),
            xytext=(x[-1] + 0.12, bl.iloc[-1] + 0.4), color="#a87200",
            fontsize=10.5, fontweight="bold", va="bottom")
ax.set_xticks(x); ax.set_xticklabels([str(c) for c in counts])
ax.set_xlim(-0.3, len(counts) + 0.75); ax.set_ylim(0, 48)
ax.set_xlabel("Complete failure histories available for training (fleet units)")
ax.set_ylabel("Prediction error  (RMSE, cycles — lower is better)")
ax.set_title("Fewer failures needed: the foundation model leads at every fleet size",
             loc="left", pad=14, fontweight="bold")
ax.text(0, 1.015, f"Turbofan benchmark {HEADLINE_DATASET} "
        f"({max(counts)} units, 6 operating conditions) · mean of 5 runs ± 1 std",
        transform=ax.transAxes, fontsize=10, color=INK2)
leg = ax.legend(loc="upper right", frameon=True, fontsize=10, framealpha=1.0,
                edgecolor="white", facecolor="white")
leg.set_zorder(10)
style(ax); fig.tight_layout(); save(fig, "fig1_data_efficiency"); plt.show()

## Fig 2 — Error by distance-to-failure (lead time)

In [ ]:
# Fig 2 -- error by distance-to-failure (all-cycles horizon eval, full fleet).
full_n = hz[hz.dataset == HEADLINE_DATASET].n_units.max()
hb = hz[(hz.dataset == HEADLINE_DATASET) & (hz.n_units == full_n) &
        (hz.bin_lo != "all")].copy()
hb["bin"] = hb.bin_lo.astype(float).astype(int).astype(str) + "-" + \
    hb.bin_hi.replace("inf", "inf").astype(str).str.replace(".0", "", regex=False)
order = ["0-25", "25-50", "50-75", "75-100", "100-125", "125-inf"]
lbls = ["0–25\n(imminent)", "25–50", "50–75", "75–100",
        "100–125", "≥125\n(healthy)"]
chb = (hb[(hb.model == "chronos-2_mlp") & (hb.loss == "corn")]
       .groupby("bin").rmse_clipped.mean().reindex(order))
gbb = hb[hb.model == "gbm"].groupby("bin").rmse_clipped.mean().reindex(order)

fig, ax = plt.subplots(figsize=(9.2, 5.4), dpi=110)
xb = np.arange(len(order)); w = 0.38
ax.bar(xb - w / 2, chb.values, w * 0.94, color=BLUE, label="Foundation model", zorder=3)
ax.bar(xb + w / 2, gbb.values, w * 0.94, color=YELLOW,
       label="From-scratch (GBM, strongest baseline)", zorder=3)
for i, key in enumerate(order):
    if key in ("25-50", "50-75"):
        ax.annotate(f"{gbb[key] / chb[key]:.1f}× better",
                    xy=(xb[i], max(chb[key], gbb[key]) + 0.8),
                    ha="center", fontsize=10.5, fontweight="bold", color=INK)
ax.axvspan(0.5, 2.5, color=GREEN, alpha=0.055, zorder=1)
top = ax.get_ylim()[1]
ax.set_ylim(0, max(27, top))
ax.text(1.5, ax.get_ylim()[1] * 0.93, "the planning window", ha="center",
        fontsize=10.5, color=GREEN, fontweight="bold")
ax.text(1.5, ax.get_ylim()[1] * 0.885, "(time to schedule maintenance)",
        ha="center", fontsize=9.5, color=GREEN)
ax.set_xticks(xb); ax.set_xticklabels(lbls, fontsize=10)
ax.set_xlabel("How far from failure the prediction is made (cycles of remaining life)")
ax.set_ylabel("Prediction error  (RMSE, cycles — lower is better)")
ax.set_title("The advantage is largest where it buys lead time",
             loc="left", pad=14, fontweight="bold")
ax.text(0, 1.015, f"Turbofan benchmark {HEADLINE_DATASET} · every test cycle "
        "scored · mean of 5 runs · full fleet",
        transform=ax.transAxes, fontsize=10, color=INK2)
ax.legend(loc="upper right", frameon=False, fontsize=10)
style(ax); fig.tight_layout(); save(fig, "fig2_lead_time"); plt.show()

## Fig 3 — XJTU-SY: vibration is a late-warning signal

In [ ]:
# Fig 3 -- XJTU-SY: models detect bearing failure LATE and too coarsely to
# plan on. Each band compares the best model against the do-nothing policy
# ("assume healthy": always predict the cap). In the healthy band the capped
# target IS the constant cap, so do-nothing scores 0 by definition -- which is
# why model error must be read against it, not in isolation.
# Needs the small XJTU per-cycle predictions file (~1 MB).
pred_files = sorted(RESULTS_DIR.glob("XJTU-SY_*_horizon_predictions.csv"))
xp = pd.read_csv(pred_files[0])
CAP = float(xp.max_rul.iloc[0]) if "max_rul" in xp else 125.0
bands = [(0, 75, "<75 min\nto failure"), (75, 125, "75–125 min"),
         (125, np.inf, "healthy\n(≥125 min)")]
rows, dn, band_n = [], {}, {}
for lo, hi, name in bands:
    m = (xp.true_rul >= lo) & (xp.true_rul < hi)
    yt = np.clip(xp.true_rul[m], None, CAP)
    dn[name] = float(np.sqrt(np.mean((CAP - yt) ** 2)))
    band_n[name] = int(m.sum() // xp.groupby(["model", "loss", "seed"]).ngroups)
    for (mo, lo_, s), g in xp[m].groupby(["model", "loss", "seed"]):
        yc = np.clip(g.true_rul, None, CAP)
        rows.append(dict(arm=f"{mo}/{lo_}", seed=s, band=name,
                         rmse=float(np.sqrt(np.mean((g.pred - yc) ** 2))),
                         bias=float((g.pred - yc).mean())))
xr = pd.DataFrame(rows)
best = xr.groupby(["arm", "band"]).rmse.mean().groupby("band").min()
names = [b[2] for b in bands]
nb_ = xr[xr.band == names[0]].groupby("arm").bias.mean()  # near-failure bias

fig, ax = plt.subplots(figsize=(9.0, 5.4), dpi=110)
xpos = np.arange(3); w = 0.36
ax.bar(xpos - w / 2, [dn[n] for n in names], w * 0.94, color="#c9c6bd",
       label="Do nothing: always assume 'healthy'", zorder=3)
ax.bar(xpos + w / 2, [best[n] for n in names], w * 0.94, color=BLUE,
       label="Best model (best of all 4, per band)", zorder=3)
for i, n in enumerate(names):
    ax.annotate(f"±{dn[n]:.0f}", xy=(xpos[i] - w / 2, dn[n] + 1.5), ha="center",
                fontsize=11, fontweight="bold", color=INK2)
    ax.annotate(f"±{best[n]:.0f} min", xy=(xpos[i] + w / 2, best[n] + 1.5),
                ha="center", fontsize=11, fontweight="bold", color=INK)
ax.annotate("0 by definition —\n'healthy' is the capped answer",
            xy=(xpos[2] - w / 2 + 0.02, 2.0), xytext=(xpos[2] - w / 2 - 0.42, 20),
            fontsize=9.5, color=INK2, ha="center",
            arrowprops=dict(arrowstyle="-", color=INK2, lw=0.8,
                            relpos=(0.85, 0.0)))
ax.annotate("detection exists…\nbut ±41 min on a <75-min\nwindow is not plannable",
            xy=(xpos[0] + w / 2, best[names[0]] + 6), fontsize=9.5, color=INK,
            ha="center", va="bottom")
ax.set_xticks(xpos); ax.set_xticklabels(
    [f"{n}\n({band_n[n]} readings/model)" for n in names], fontsize=10)
ax.set_ylim(0, max(dn.values()) + 22)
ax.set_ylabel("Remaining-life error  (RMSE, minutes)")
ax.set_title("Bearings: vibration detects failure late — too coarsely to plan on",
             loc="left", pad=14, fontweight="bold")
ax.text(0, 1.015, "XJTU-SY run-to-failure bearings · every test reading scored "
        "· mean of 5 runs", transform=ax.transAxes, fontsize=10, color=INK2)
ax.text(0, -0.22, "Near failure every model also OVER-estimates remaining life "
        f"(bias {nb_.min():+.0f} to {nb_.max():+.0f} min — predictions lag the "
        "collapse).\nContrast with the turbofan figure, where error shrinks "
        "toward failure: gradual degradation is predictable, cliff-shaped is "
        "not.\nThat contrast is the case for an earlier-warning modality "
        "(acoustic emission) on flat-then-cliff components.",
        transform=ax.transAxes, fontsize=10, color=INK2, va="top")
ax.legend(loc="upper right", frameon=False, fontsize=10)
style(ax); fig.tight_layout(); save(fig, "fig3_vibration_late_warning"); plt.show()

## Fig 4 — Small-fleet robustness census (N-CMAPSS)

In [ ]:
# Fig 4 -- robustness census on the small N-CMAPSS fleets, from the horizon
# eval's overall ('all') rows: a run "failed" if its all-cycle RMSE > 25
# (every diverged/constant-output run trips this).
nc = hz[hz.dataset.str.startswith("DS") & (hz.bin_lo == "all")].copy()
nc["arm"] = nc.model + "/" + nc.loss
census_arms = {
    "chronos-2_mlp/corn": "Foundation + ordinal head (CORN)  ← recommended",
    "lstm/native": "From-scratch LSTM",
    "gbm/native": "From-scratch GBM",
    "chronos-2_mlp/mse": "Foundation + standard loss (MSE)",
}
cen = (nc[nc.arm.isin(census_arms)].groupby("arm")
       .agg(total=("rmse_clipped", "size"),
            bad=("rmse_clipped", lambda s: int((s > 25).sum()))))
cen = cen.loc[list(census_arms)].rename(index=census_arms)
cen["pct"] = cen.bad / cen.total * 100
cen = cen.iloc[::-1]

fig, ax = plt.subplots(figsize=(9.0, 4.6), dpi=110)
cols = [ORANGE if "MSE" in a else (BLUE if "recommended" in a else "#c9c6bd")
        for a in cen.index]
ax.barh(np.arange(len(cen)), cen.pct, 0.55, color=cols, zorder=3)
for i, (a, r) in enumerate(cen.iterrows()):
    ax.annotate(f"{int(r.bad)}/{int(r.total)} runs  ({r.pct:.0f}%)",
                xy=(r.pct + 0.8, i), va="center", fontsize=11,
                fontweight="bold", color=INK)
ax.set_yticks(np.arange(len(cen))); ax.set_yticklabels(cen.index, fontsize=11)
ax.set_xlabel("Training runs that failed to produce a usable model "
              f"(of {int(cen.total.max())} small-fleet runs)")
ax.set_xlim(0, 55)
ax.set_title("With scarce failure data, configuration decides reliability",
             loc="left", pad=14, fontweight="bold")
ax.text(0, 1.02, "9 realistic small fleets (6–9 machines) × 5 repeats · "
        "failed = all-cycle RMSE > 25 (diverged or constant output)",
        transform=ax.transAxes, fontsize=10, color=INK2)
style(ax); ax.grid(axis="x"); ax.grid(axis="y", visible=False)
fig.tight_layout(); save(fig, "fig4_robustness"); plt.show()

## Backup — same pattern on every C-MAPSS dataset

In [ ]:
# Backup slides: fig-1 small multiples for the other C-MAPSS datasets, so
# "did you only show the good dataset?" has a ready answer (pattern holds 4/4).
others = [ds for ds in ["FD001", "FD003", "FD004"] if ds in set(res.dataset)]
fig, axes = plt.subplots(1, len(others), figsize=(4.6 * len(others), 3.8),
                         dpi=110, sharey=True)
for ax, ds in zip(np.atleast_1d(axes), others):
    dd = res[res.dataset == ds]
    cts = sorted(dd.n_units.unique())
    gg = dd.groupby(["arm", "n_units"]).rmse_clipped.mean()
    chm = gg.loc["chronos-2_mlp/corn"].reindex(cts)
    blm = (dd[dd.arm.isin(BASELINE_ARMS)].groupby(["arm", "n_units"])
           .rmse_clipped.mean().groupby("n_units").min().reindex(cts))
    xx = np.arange(len(cts))
    ax.plot(xx, chm.values, color=BLUE, lw=2.2, marker="o", ms=5)
    ax.plot(xx, blm.values, color=YELLOW, lw=1.8, ls="--", marker="s", ms=5)
    ax.set_xticks(xx); ax.set_xticklabels([str(c) for c in cts], fontsize=9)
    ax.set_title(ds, fontsize=12, fontweight="bold", loc="left")
    ax.set_xlabel("failure histories"); style(ax)
np.atleast_1d(axes)[0].set_ylabel("RMSE (cycles)")
fig.suptitle("Backup: the same pattern on every C-MAPSS dataset "
             "(blue = foundation, yellow = strongest from-scratch)",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.93)); save(fig, "fig1b_backup_small_multiples")
plt.show()